In [1]:
"""
Remove duplicate prompts from step_bench dataset
"""

import pandas as pd
import numpy as np
import os
from collections import defaultdict

def remove_duplicates_step_bench(input_path, output_path):
    """
    Remove duplicate prompts from step_bench dataset
    
    Args:
        input_path: Path to the original dataset
        output_path: Path to save the deduplicated dataset
    """
    print(f"Loading dataset from: {input_path}")
    df = pd.read_parquet(input_path)
    
    print(f"Original dataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    
    # Convert prompts to strings for deduplication
    prompt_strings = []
    for i, prompt in enumerate(df['prompt']):
        if isinstance(prompt, np.ndarray):
            prompt_str = str(prompt.tolist())  # Convert array to string
        else:
            prompt_str = str(prompt)
        prompt_strings.append(prompt_str)
    
    # Find unique prompts (keep first occurrence)
    seen_prompts = set()
    unique_indices = []
    duplicate_count = 0
    
    for i, prompt_str in enumerate(prompt_strings):
        if prompt_str not in seen_prompts:
            seen_prompts.add(prompt_str)
            unique_indices.append(i)
        else:
            duplicate_count += 1
    
    print(f"Found {duplicate_count} duplicate prompts")
    print(f"Keeping {len(unique_indices)} unique prompts")
    
    # Create deduplicated dataframe
    df_unique = df.iloc[unique_indices].copy()
    df_unique = df_unique.reset_index(drop=True)
    
    print(f"Deduplicated dataset shape: {df_unique.shape}")
    
    # Save the deduplicated dataset
    print(f"Saving deduplicated dataset to: {output_path}")
    df_unique.to_parquet(output_path, index=False)
    
    # Statistics
    print("\n=== Deduplication Summary ===")
    print(f"Original samples: {len(df)}")
    print(f"Unique samples: {len(df_unique)}")
    print(f"Removed duplicates: {duplicate_count}")
    print(f"Deduplication rate: {duplicate_count/len(df)*100:.1f}%")
    
    return df_unique

def analyze_duplicates(input_path):
    """
    Analyze duplicate patterns in the dataset
    """
    print("=== Analyzing duplicate patterns ===")
    df = pd.read_parquet(input_path)
    
    # Convert prompts to strings
    prompt_strings = []
    for prompt in df['prompt']:
        if isinstance(prompt, np.ndarray):
            prompt_str = str(prompt.tolist())
        else:
            prompt_str = str(prompt)
        prompt_strings.append(prompt_str)
    
    # Count occurrences
    prompt_counts = defaultdict(int)
    for prompt_str in prompt_strings:
        prompt_counts[prompt_str] += 1
    
    # Analyze distribution
    count_distribution = defaultdict(int)
    for count in prompt_counts.values():
        count_distribution[count] += 1
    
    print("Prompt occurrence distribution:")
    for count, num_prompts in sorted(count_distribution.items()):
        print(f"  {num_prompts} prompts appear {count} time(s)")
    
    # Show examples of duplicates
    print("\nExamples of prompts with multiple occurrences:")
    duplicate_examples = 0
    for prompt_str, count in prompt_counts.items():
        if count > 1 and duplicate_examples < 3:
            print(f"  Prompt appears {count} times: {prompt_str[:100]}...")
            duplicate_examples += 1

if __name__ == "__main__":
    # Paths
    input_path = "/ibex/project/c2261/wenxuan/projects/dev_verl/data/my_data_new/step_bench/step_bench_modified.parquet"
    output_path = "/ibex/project/c2261/wenxuan/projects/dev_verl/data/my_data_new/step_bench/step_bench_deduplicated.parquet"
    
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Analyze duplicates first
    analyze_duplicates(input_path)
    
    print("\n" + "="*50)
    
    # Remove duplicates
    df_unique = remove_duplicates_step_bench(input_path, output_path)
    
    print(f"\n✅ Successfully created deduplicated dataset!")
    print(f"Original file: {input_path}")
    print(f"New file: {output_path}") 

=== Analyzing duplicate patterns ===
Prompt occurrence distribution:
  785 prompts appear 1 time(s)
  32 prompts appear 2 time(s)
  10 prompts appear 3 time(s)
  4 prompts appear 4 time(s)
  1 prompts appear 5 time(s)

Examples of prompts with multiple occurrences:
  Prompt appears 2 times: [{'content': '\n    You are an expert in evaluating the correctness of math solutions.\n    You will...
  Prompt appears 4 times: [{'content': '\n    You are an expert in evaluating the correctness of math solutions.\n    You will...
  Prompt appears 2 times: [{'content': '\n    You are an expert in evaluating the correctness of math solutions.\n    You will...

Loading dataset from: /ibex/project/c2261/wenxuan/projects/dev_verl/data/my_data_new/step_bench/step_bench_modified.parquet
Original dataset shape: (900, 5)
Columns: ['data_source', 'prompt', 'ability', 'reward_model', 'extra_info']
Found 68 duplicate prompts
Keeping 832 unique prompts
Deduplicated dataset shape: (832, 5)
Saving deduplicated